# C2U in RAG: Compositional Concept Unlearning
### Full Pipeline on Kaggle GPU (TOFU, MNLI, Amazon, ArXiv)
Notebook này thực thi toàn trình thực nghiệm C2U theo bài báo gốc:
1. **Encode Cache:** Trích xuất hBcnormalized embeddings qua Frozen Encoders.
2. **Train ACR:** Huấn luyện từ điển nguyên tử  \in \mathbb{R}^{N 	imes d}$ và projection head.
3. **C2U Operator:** Áp dụng 4 bước chỉnh sửa vector truy vấn online.
4. **Evaluation:** Đánh giá Top-5 Retrieval (Target Hit@5 Drop, Control Utility Drop, Target Exposure).

In [ ]:
# CELL 1: Kiểm tra phần cứng GPU & Cài đặt thư viện phụ thuộc
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

!pip install -q sentence-transformers tabulate

In [ ]:
# CELL 2: Giải nén Mã nguồn & Dữ liệu từ Dataset c2u-bundle
import os, zipfile, shutil

# Tìm và giải nén bundle từ Kaggle input vào /kaggle/working
bundle_candidates = [
    "/kaggle/input/c2u-bundle/c2u_full_bundle.zip",
    "/kaggle/input/c2u-splits/c2u_full_bundle.zip",
    "/kaggle/input/c2u-splits/c2u_splits.zip"
]

extracted = False
for path in bundle_candidates:
    if os.path.exists(path):
        print(f"[Setup] Giải nén gói: {path}...")
        with zipfile.ZipFile(path, "r") as zf:
            zf.extractall("/kaggle/working")
        extracted = True
        break

if not extracted:
    # Nếu upload dạng thư mục không nén
    print("[Setup] Kiểm tra thư mục input trực tiếp...")

!ls -lh /kaggle/working

In [ ]:
# CELL 3: Chạy Kiểm thử Sanity Check (Dry-run) trên GPU
# Xác nhận 100% gradient ACR, lstsq, và vùng tin cậy Trust-Region hoạt động hoàn hảo
!python /kaggle/working/test_dry_run.py

In [ ]:
# CELL 4: Chạy Pipeline Thực nghiệm C2U
# Chọn tập dữ liệu: "tofu", "mnli", "amazon", "arxiv" hoặc "all"
# Chọn mô hình encoder: 
#   - sentence-transformers/all-MiniLM-L6-v2 (d=384, N=768)
#   - BAAI/bge-base-en-v1.5 (d=768, N=1536)
#   - intfloat/e5-base-v2 (d=768, N=1536)

DATASET = "tofu"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EPOCHS = 15
BATCH_SIZE = 128

!python /kaggle/working/run_kaggle_pipeline.py \
    --dataset {DATASET} \
    --model_name {MODEL_NAME} \
    --work_dir /kaggle/working \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE}

In [ ]:
# CELL 5: Trực quan hóa Kết quả Thực nghiệm qua Biểu đồ
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

csv_path = "/kaggle/working/c2u_final_results.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("=== BẢNG KẾT QUẢ ĐẦY ĐỦ ===")
    display(df)

    # Vẽ đồ thị so sánh
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    sns.set_theme(style="whitegrid")

    # 1. Target Hit@5 Drop
    sns.barplot(data=df[df["Method"] == "C2U"], x="Dataset", y="Target_Hit@5_Drop", hue="Model", ax=axes[0], palette="Blues_d")
    axes[0].set_title("Target Hit@5 Drop (Càng cao càng tốt)", fontsize=12, fontweight="bold")
    axes[0].set_ylabel("Mức giảm tỷ lệ Target (%)")

    # 2. Control Utility Drop
    sns.barplot(data=df[df["Method"] == "C2U"], x="Dataset", y="Control_Utility_Drop", hue="Model", ax=axes[1], palette="Greens_d")
    axes[1].axhline(0, color="red", linestyle="--", alpha=0.7)
    axes[1].set_title("Control Utility Drop (Mục tiêu <= 0)", fontsize=12, fontweight="bold")
    axes[1].set_ylabel("Mức suy giảm tiện ích Control")

    # 3. Target Exposure Change
    sns.barplot(data=df[df["Method"] == "C2U"], x="Dataset", y="TE_Change", hue="Model", ax=axes[2], palette="Oranges_d")
    axes[2].axhline(0, color="red", linestyle="--", alpha=0.7)
    axes[2].set_title("Target Exposure Change in Control (<= 0)", fontsize=12, fontweight="bold")
    axes[2].set_ylabel("Thay đổi độ phơi bày")

    plt.tight_layout()
    plt.savefig("/kaggle/working/c2u_evaluation_plot.png", dpi=300)
    plt.show()
    print("[Plot] Đã lưu biểu đồ vào /kaggle/working/c2u_evaluation_plot.png")
else:
    print("Chưa tìm thấy file kết quả CSV. Vui lòng chạy Cell 4 trước.")

In [ ]:
# CELL 6: Tải về Checkpoints và File Báo cáo Kết quả CSV
from IPython.display import FileLink

# Hiển thị đường dẫn tải trực tiếp file CSV
print("Click vào liên kết dưới đây để tải kết quả về máy:")
display(FileLink("c2u_final_results.csv"))
if os.path.exists("c2u_evaluation_plot.png"):
    display(FileLink("c2u_evaluation_plot.png"))